# Fine-Tuning Whisper Large-v2 for Nigerian Pidgin ASR

This notebook fine-tunes **OpenAI Whisper large-v2** for automatic speech recognition (ASR) on the **Nigerian Pidgin 1.0** dataset using **Parameter-Efficient Fine-Tuning (PEFT) with LoRA**.

> **Important implementation note:** The original training run used **LoRA fine-tuning on the standard Whisper large-v2 checkpoint**. It did **not** load the training model with BitsAndBytes INT8 quantization. **8-bit quantization is used only when loading the fine-tuned model for inference**, where it reduces memory requirements.

The notebook is designed for a Google Colab GPU environment and covers the complete workflow:

1. Environment setup
2. Dataset preparation
3. Whisper preprocessing
4. LoRA configuration and training
5. Evaluation with Word Error Rate (WER)
6. 8-bit inference
7. An interactive Gradio demo

The workflow was adapted from the Hugging Face Whisper fine-tuning approach and tailored to Nigerian Pidgin ASR.


## 1. Environment Setup

The original notebook contained several repeated package-installation cells and installed development versions directly from GitHub. For a public project, we keep setup in one place and avoid depending on a moving `main` branch.

> For exact environment reproducibility, the final GitHub repository should also include a pinned `requirements.txt` generated from the working environment.


In [ ]:
# Install the packages used by this notebook.
# For exact reproducibility, pin the tested versions in requirements.txt.

!pip install -q \
    transformers \
    peft \
    datasets \
    evaluate \
    jiwer \
    accelerate \
    bitsandbytes \
    librosa \
    gradio \
    huggingface_hub


### Optional: verify the runtime

Whisper large-v2 is a large model. A CUDA-enabled GPU is strongly recommended for training and evaluation.


In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2. Hugging Face Authentication

Authentication is required only if you want to upload the trained LoRA adapter to the Hugging Face Hub. Never place an access token directly in the notebook or commit it to GitHub.


In [ ]:
from getpass import getpass
from huggingface_hub import login

# Uncomment when Hub authentication is required.
# hf_token = getpass("Enter your Hugging Face token: ")
# login(token=hf_token)


## 3. Project Configuration

The dataset is Nigerian Pidgin, and the target task is speech transcription.

### Language-prompt note

The dataset contains **Nigerian Pidgin** transcripts. In the original workflow, the Whisper tokenizer/processor was configured with `English` because Nigerian Pidgin was not exposed as a Whisper decoder-language option in the interface used at the time. The dataset language code `pcm` is retained as project metadata.

This setting is a **decoder prompt**, not a claim that the target transcripts are English. The training labels remain the Nigerian Pidgin sentences from the dataset, and the final model is evaluated against Nigerian Pidgin references.


In [ ]:
import os

# Select the first CUDA device in Colab.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

MODEL_NAME = "openai/whisper-large-v2"
DATASET_NAME = "asr-nigerian-pidgin/nigerian-pidgin-1.0"
DATASET_LANGUAGE_CODE = "pcm"  # Nigerian Pidgin language code used by the dataset
WHISPER_PROMPT_LANGUAGE = "English"  # Decoder prompt used by the original workflow
TASK = "transcribe"

# Hugging Face Hub repository used by the original training run.
HF_REPO_ID = "Koredegreatnessnow/openai-whisper-large-v2-lora-colab"

OUTPUT_DIR = "whisper-large-v2-nigerian-pidgin-lora"


## 4. Load the Nigerian Pidgin Dataset

The dataset provides `train`, `validation`, and `test` splits. Following the original experiment, the training and validation splits are combined into a single training set, while the original test split is retained for final evaluation.

This gives **3,385 training examples** (2,708 original training + 677 validation) and **892 test examples** in the recorded experiment.

Because the validation split is combined into training, the test set is the independent evaluation set reported below.


In [ ]:
from datasets import load_dataset, DatasetDict

dataset = load_dataset(DATASET_NAME)

common_voice = DatasetDict({
    "train": load_dataset(DATASET_NAME, split="train+validation"),
    "test": load_dataset(DATASET_NAME, split="test"),
})

print(common_voice)


In [ ]:
# Inspect one training example.
common_voice["train"][0]


## 5. Prepare the Whisper Feature Extractor and Tokenizer

Whisper expects audio at **16 kHz**. The dataset audio is therefore resampled to 16 kHz before feature extraction. The feature extractor converts the waveform into Whisper's log-Mel input features, while the tokenizer converts the target transcript into token IDs.


In [ ]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)

tokenizer = WhisperTokenizer.from_pretrained(
    MODEL_NAME,
    language=WHISPER_PROMPT_LANGUAGE,
    task=TASK,
)

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language=WHISPER_PROMPT_LANGUAGE,
    task=TASK,
)


In [ ]:
from datasets import Audio

# Resample audio to the 16 kHz sampling rate expected by Whisper.
common_voice = common_voice.cast_column("audio", Audio(sampling_rate=16000))

print(common_voice["train"][0]["audio"]["sampling_rate"])


## 6. Convert Audio and Transcripts into Model Inputs

For each example:

- the audio is loaded at 16 kHz;
- Whisper computes log-Mel spectrogram features; and
- the Nigerian Pidgin transcript is tokenized into label IDs.


In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]

    # Convert waveform to Whisper input features.
    batch["input_features"] = feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
    ).input_features[0]

    # Tokenize the Nigerian Pidgin transcript.
    batch["labels"] = tokenizer(batch["sentence"]).input_ids

    return batch


In [ ]:
# Process the complete dataset.
# num_proc=1 is retained from the original working experiment for stability in Colab.
common_voice = common_voice.map(
    prepare_dataset,
    remove_columns=common_voice.column_names["train"],
    num_proc=1,
)

print(common_voice["train"])


## 7. Data Collator

Audio features and text labels have different sequence lengths, so they must be padded separately. The collator pads the inputs and replaces padded label positions with `-100`, which tells the loss function to ignore those positions.


In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(
        self,
        features: List[Dict[str, Union[List[int], torch.Tensor]]],
    ) -> Dict[str, torch.Tensor]:
        input_features = [
            {"input_features": feature["input_features"]}
            for feature in features
        ]
        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt",
        )

        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100,
        )

        # Whisper adds the BOS token during generation; avoid duplicating it.
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)


## 8. Evaluation Metric: Word Error Rate (WER)

**Word Error Rate (WER)** measures the difference between the reference transcript and the model prediction using substitutions, deletions, and insertions. Lower WER indicates fewer word-level transcription errors.

For this project, WER is calculated on the held-out Nigerian Pidgin test set.


In [ ]:
import evaluate

metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids.copy()

    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(
        predictions=pred_str,
        references=label_str,
    )

    return {"wer": wer}


## 9. Load the Whisper Large-v2 Base Model

The base Whisper large-v2 checkpoint is loaded in its standard form for training.

> **This is where the original notebook differs from the title it previously used:** no `BitsAndBytesConfig(load_in_8bit=True)` is applied here. Therefore, the training run should be described as **LoRA fine-tuning**, not INT8/BNB training.


In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

# Do not force decoder language/task tokens during training.
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []


## 10. Apply LoRA (Parameter-Efficient Fine-Tuning)

LoRA adds small trainable low-rank adapter matrices to selected layers while keeping the original Whisper parameters frozen. In this experiment, LoRA is applied to the attention `q_proj` and `v_proj` modules.

The original configuration was: `r=32`, `lora_alpha=64`, `lora_dropout=0.05`, and `bias="none"`.


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 11. Training Configuration

The configuration below reflects the original working training run. The small per-device batch size was used for GPU-memory constraints, while gradient accumulation increases the effective batch size. One training epoch was used in the recorded experiment.

`remove_unused_columns=False` and `label_names=["labels"]` are retained because the PEFT-wrapped model requires the label information to be passed explicitly in this training setup.

The built-in Trainer generation-based evaluation is not enabled here; evaluation is performed separately after training with a manual generation loop.


In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-3,
    warmup_steps=50,
    num_train_epochs=1,
    fp16=False,
    per_device_eval_batch_size=1,
    generation_max_length=128,
    logging_steps=25,
    remove_unused_columns=False,
    label_names=["labels"],
)


## 12. Train the LoRA Adapter

The independent test set is kept out of the training/evaluation loop. The trainer is used only for LoRA training; the test set is evaluated separately afterward.


In [ ]:
from transformers import Seq2SeqTrainer

model.config.use_cache = False

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=common_voice["train"],
    # Keep the independent test set out of Trainer evaluation.
    data_collator=data_collator,
)

train_result = trainer.train()
print(train_result)


### Recorded training result

The notebook's original completed run recorded:

- **Global steps:** 424
- **Training loss:** 6.7274
- **Epoch:** 1.0
- **Training runtime:** approximately 132 minutes

These figures are retained as historical results from the original run; re-running the notebook may produce different values depending on software versions, hardware, and randomness.


## 13. Save or Upload the LoRA Adapter

The trained model is a PEFT adapter rather than a full copy of Whisper large-v2. This keeps the resulting adapter substantially smaller than the full base model.

The following cell uploads the adapter to the Hugging Face Hub if you are authenticated and have permission to write to the repository.


In [ ]:
# Upload the trained LoRA adapter to the Hugging Face Hub.
# Uncomment after authenticating with the Hub.

# model.push_to_hub(HF_REPO_ID)
# print(f"Uploaded adapter to: {HF_REPO_ID}")


## 14. Load the Fine-Tuned Adapter for Evaluation

For evaluation, the base Whisper model is loaded and the saved LoRA adapter is attached. The base model is kept in its standard form here so that the evaluation procedure remains faithful to the original experiment.


In [ ]:
from peft import PeftModel, PeftConfig
from transformers import WhisperForConditionalGeneration

# If the adapter is stored locally, replace HF_REPO_ID with its local path.
peft_model_id = HF_REPO_ID

peft_config = PeftConfig.from_pretrained(peft_model_id)

model = WhisperForConditionalGeneration.from_pretrained(
    peft_config.base_model_name_or_path
)
model = PeftModel.from_pretrained(model, peft_model_id)

model.config.use_cache = True


## 15. Evaluate on the Nigerian Pidgin Test Set

The evaluation loop generates predictions for all **892 test examples** without using any reference-transcript tokens as decoder inputs. A fixed Whisper decoder prompt is used instead. This keeps the test set independent and avoids reference leakage.


In [ ]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import numpy as np
import gc
import torch

metric = evaluate.load("wer")

eval_dataloader = DataLoader(
    common_voice["test"],
    batch_size=1,
    collate_fn=data_collator,
)

if torch.cuda.is_available():
    model = model.to("cuda")

# Use the same fixed decoder prompt as the project workflow.
# This is a model-side prompt and does not use reference transcript tokens.
forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=WHISPER_PROMPT_LANGUAGE,
    task=TASK,
)

model.eval()
for batch in tqdm(eval_dataloader, desc="Evaluating"):
    with torch.no_grad():
        device = "cuda" if torch.cuda.is_available() else "cpu"
        generated_tokens = model.generate(
            input_features=batch["input_features"].to(device),
            forced_decoder_ids=forced_decoder_ids,
            max_length=255,
        ).cpu().numpy()

        labels = batch["labels"].cpu().numpy()
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

        decoded_preds = tokenizer.batch_decode(
            generated_tokens, skip_special_tokens=True
        )
        decoded_labels = tokenizer.batch_decode(
            labels, skip_special_tokens=True
        )

        metric.add_batch(
            predictions=decoded_preds,
            references=decoded_labels,
        )

        del generated_tokens, labels, batch
        gc.collect()

wer = 100 * metric.compute()
print(f"Corrected WER = {wer:.2f}%")


### Historical evaluation result

The original notebook recorded **WER = 20.40%** on the 892-example test set. However, that original evaluation loop supplied the first four reference-label tokens as `decoder_input_ids`, which means the reported value should be treated as a **historical result of the original implementation**, not as the clean benchmark for this repository.

The corrected evaluation cell above removes that reference-token leakage. Because the corrected evaluation has not been run in this environment, this notebook does **not** claim a new corrected WER. Run the corrected cell to obtain the repository benchmark.


## 16. 8-bit Quantized Inference

The original notebook uses **BitsAndBytes INT8 quantization here, during inference—not during training**. The base Whisper model is loaded in 8-bit and the LoRA adapter is then attached.

This reduces inference memory requirements and is useful when deploying the fine-tuned adapter in a constrained GPU environment.


In [ ]:
from transformers import (
    WhisperForConditionalGeneration,
    WhisperTokenizer,
    WhisperProcessor,
    BitsAndBytesConfig,
)
from peft import PeftModel, PeftConfig

peft_config = PeftConfig.from_pretrained(HF_REPO_ID)

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
)

quantized_model = WhisperForConditionalGeneration.from_pretrained(
    peft_config.base_model_name_or_path,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

quantized_model = PeftModel.from_pretrained(
    quantized_model,
    HF_REPO_ID,
)

quantized_tokenizer = WhisperTokenizer.from_pretrained(
    peft_config.base_model_name_or_path,
    language=WHISPER_PROMPT_LANGUAGE,
    task=TASK,
)

quantized_processor = WhisperProcessor.from_pretrained(
    peft_config.base_model_name_or_path,
    language=WHISPER_PROMPT_LANGUAGE,
    task=TASK,
)

quantized_model.eval()


## 17. Simple Audio Transcription Function

The function below accepts a local audio file, extracts Whisper features, and generates a transcription using the quantized base model plus the LoRA adapter.


In [ ]:
import librosa

def transcribe_audio(audio_path, max_length=255):
    audio, sampling_rate = librosa.load(audio_path, sr=16000, mono=True)

    inputs = quantized_processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
    )

    # Quantized models manage their device placement through device_map.
    device = next(quantized_model.parameters()).device
    input_features = inputs.input_features.to(device)

    forced_decoder_ids = quantized_processor.get_decoder_prompt_ids(
        language=WHISPER_PROMPT_LANGUAGE,
        task=TASK,
    )

    with torch.no_grad():
        generated_tokens = quantized_model.generate(
            input_features=input_features,
            forced_decoder_ids=forced_decoder_ids,
            max_length=max_length,
        )

    return quantized_tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True,
    )[0]


## 18. Interactive Gradio Demo

The demo provides two input options:

- upload an audio file; or
- record speech directly from the microphone.

The output is the model's Nigerian Pidgin transcription.


In [ ]:
import gradio as gr

def gradio_transcribe(audio_path):
    if audio_path is None:
        return "No audio provided."
    return transcribe_audio(audio_path)

iface = gr.Interface(
    fn=gradio_transcribe,
    inputs=gr.Audio(
        sources=["upload", "microphone"],
        type="filepath",
    ),
    outputs=gr.Textbox(label="Transcription"),
    title="Nigerian Pidgin Speech-to-Text",
    description=(
        "Upload an audio file or record speech with your microphone "
        "to generate a Nigerian Pidgin transcription."
    ),
)

iface.launch(share=True)


## 19. Limitations and Reproducibility Notes

- The training configuration reflects a **one-epoch** experiment performed under GPU-memory and compute constraints.
- The original training and validation splits were combined; therefore, the test split is the independent evaluation set.
- The original Whisper decoder prompt used in this workflow is `English`, while the training and evaluation transcripts are Nigerian Pidgin. This reflects the tokenizer configuration available in the original workflow and should not be interpreted as relabelling the dataset as English.
- The historical **20.40% WER** should not be presented as the clean benchmark because the original evaluation loop used reference transcript tokens as decoder inputs. The corrected evaluation avoids that leakage.
- Exact reproduction requires matching the software environment and hardware as closely as possible. The final GitHub repository should include a pinned `requirements.txt`.
- The interactive Gradio `share=True` URL is temporary and should not be treated as a permanent deployment.


## 20. Project Summary

This project demonstrates a parameter-efficient approach to adapting **Whisper large-v2** for **Nigerian Pidgin automatic speech recognition**. The training stage uses **LoRA**, while **8-bit BitsAndBytes quantization is applied during inference** to reduce memory requirements.

The original experiment trained for one epoch on 3,385 examples and evaluated on 892 test examples. The original notebook recorded a historical WER of **20.40%**, but the repository now uses a corrected evaluation procedure that does not feed reference transcript tokens into generation.
